# ML-09 — Validation and Research Claim Audit

This notebook attacks the Week-5 model before trusting it. It compares a random row split with a client-grouped split, checks the final features for leakage, and rewrites the claim in safe decision-support language.


## 1. Two paper findings + my methodology questions

**Finding A — growing vs declining content.** FlyRank's public report says growing pages were younger on average than declining pages while average word count was nearly the same. My methodology question is: how exactly is growing/declining labeled, and does validation prevent the same client/site characteristics from crossing train/test? A random row split could overstate generalization.

**Finding B — CTR-vs-position.** FlyRank's CTR-fix logic compares CTR with position-tier expectations instead of a universal CTR threshold. My methodology question is: are those position-tier benchmarks computed on training data only before evaluating held-out pages? Full-population benchmarks would leak information across the split.

Both findings can be useful as observed patterns, but claim strength should match label construction and validation design.


In [1]:
%pip -q install duckdb huggingface_hub pandas numpy scikit-learn
import os,sys,json
from pathlib import Path
import pandas as pd,numpy as np
from google.colab import userdata
from huggingface_hub import whoami
REPO_DIR=Path('/content/Internship')
if not REPO_DIR.exists():
    !git clone https://github.com/imalik-7/Internship.git /content/Internship
os.chdir(REPO_DIR); sys.path.insert(0,str(REPO_DIR))
from work.lib.capstone_pipeline import FEATURES,LABEL,connect_warehouse,build_analysis_frame,grouped_split,random_split,train_compare,best_model_name,ensure_output_dir
HF_TOKEN=userdata.get('HF_TOKEN'); print('HF account:',whoami(token=HF_TOKEN)['name'])
con=connect_warehouse(HF_TOKEN); analysis_df=build_analysis_frame(con)
print('Rows:',len(analysis_df),'| base rate:',round(analysis_df[LABEL].mean(),3))


Cloning into '/content/Internship'...
remote: Enumerating objects: 190, done.
remote: Counting objects: 100% (190/190), done.
remote: Compressing objects: 100% (142/142), done.
remote: Total 190 (delta 79), reused 92 (delta 28), pack-reused 0 (from 0)
Receiving objects: 100% (190/190), 1.91 MiB | 6.57 MiB/s, done.
Resolving deltas: 100% (79/79), done.
HF account: imalik7


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Rows: 77540 | base rate: 0.327


## 2. My model under an honest split (before/after)

I compare a convenient random row split with the stricter client-grouped split using the exact same five features, label, models, metrics, and random seed. The grouped result is the number I keep; the gap itself shows how much client similarity may have helped the random split.


In [2]:
rtr,rte=random_split(analysis_df,test_size=0.25,random_state=42)
rcmp,rmodels,rscores,_=train_compare(rtr,rte,random_state=42)
gtr,gte=grouped_split(analysis_df,test_size=0.25,random_state=42)
gcmp,gmodels,gscores,_=train_compare(gtr,gte,random_state=42)
rb=best_model_name(rcmp); gb=best_model_name(gcmp)
before_after=pd.DataFrame([
 {'split':'Random row split','best_model':rb,'p20':float(rcmp.loc[rcmp.method.eq(rb),'precision_at_20'].iloc[0]),'p50':float(rcmp.loc[rcmp.method.eq(rb),'precision_at_50'].iloc[0]),'AP':float(rcmp.loc[rcmp.method.eq(rb),'average_precision'].iloc[0]),'base_rate':float(rte[LABEL].mean())},
 {'split':'Client-grouped split (kept)','best_model':gb,'p20':float(gcmp.loc[gcmp.method.eq(gb),'precision_at_20'].iloc[0]),'p50':float(gcmp.loc[gcmp.method.eq(gb),'precision_at_50'].iloc[0]),'AP':float(gcmp.loc[gcmp.method.eq(gb),'average_precision'].iloc[0]),'base_rate':float(gte[LABEL].mean())}
])
display(before_after.round(3))
print('Client overlap:',len(set(gtr.client_hash_id)&set(gte.client_hash_id)))
assert len(set(gtr.client_hash_id)&set(gte.client_hash_id))==0
OUT=ensure_output_dir(REPO_DIR)
with open(OUT/'w06_validation_metrics.json','w') as f: json.dump({'random_split':rcmp.to_dict(orient='records'),'grouped_split':gcmp.to_dict(orient='records'),'kept_best_model':gb},f,indent=2)


,split,best_model,p20,p50,AP,base_rate
0,Random row split,Random Forest,0.85,0.8,0.489,0.327
1,Client-grouped split (kept),Logistic Regression,0.20,0.3,0.238,0.179


Client overlap: 0


## 3. Leakage audit

Every final feature must be knowable by March 15, no label-derived sibling can be present, no existing product score/flag can be used as an input, and the split must remain grouped. I also deliberately add one outcome-derived ratio. If its score jumps, that confirms the harness catches the classic leakage trap. The leaked field is demonstration-only and is deleted from the final feature set.


In [3]:
FORBIDDEN={'impressions_outcome15','is_declining_proxy','observed_impression_loss','leaky_outcome_ratio','baseline_action_score','reason_code','action_label'}
print('Final features:',FEATURES)
print('Forbidden intersection:',set(FEATURES)&FORBIDDEN)
assert not(set(FEATURES)&FORBIDDEN)
for frame in (gtr,gte): frame['leaky_outcome_ratio']=frame['impressions_outcome15']/frame['impressions_feature15'].replace(0,np.nan)
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import average_precision_score
leaky=Pipeline([('imputer',SimpleImputer(strategy='median')),('model',DecisionTreeClassifier(max_depth=4,min_samples_leaf=50,random_state=42))])
LF=FEATURES+['leaky_outcome_ratio']; leaky.fit(gtr[LF],gtr[LABEL]); lp=leaky.predict_proba(gte[LF])[:,1]
honest_ap=float(gcmp.loc[gcmp.method.eq(gb),'average_precision'].iloc[0]); leak_ap=float(average_precision_score(gte[LABEL],lp))
print('Honest grouped AP:',round(honest_ap,3)); print('Leaked AP:',round(leak_ap,3)); print('Leakage jump:',round(leak_ap-honest_ap,3))
print('PASS: leaky_outcome_ratio is not in FEATURES and is not kept.')


Final features: ['impressions_feature15', 'clicks_feature15', 'ctr_feature15', 'avg_position_feature15', 'active_impression_days_feature15']
Forbidden intersection: set()
Honest grouped AP: 0.238
Leaked AP: 1.0
Leakage jump: 0.762
PASS: leaky_outcome_ratio is not in FEATURES and is not kept.


## 4. Claim rewrite

**Too bold:** The model identifies which pages should be refreshed and predicts which refreshes will recover traffic.

**Safe claim:** On this March 2026 evaluation slice, the learned ranking uses five pre-decision search signals to prioritize pages associated with a later impression-decline proxy. Its held-out performance is compared with the same frozen rule baseline under a client-grouped split. The output is directional decision support for human review; it does not prove that a refresh will cause recovery or reveal Google's ranking algorithm.


In [4]:
SAFE_CLAIM='On this March 2026 evaluation slice, five pre-decision search signals support a directional ranking of pages associated with a later impression-decline proxy; this is human decision support, not causal proof.'
print(SAFE_CLAIM)


On this March 2026 evaluation slice, five pre-decision search signals support a directional ranking of pages associated with a later impression-decline proxy; this is human decision support, not causal proof.


## Self-check

- [x] Two FlyRank findings are questioned constructively.
- [x] Random and client-grouped validation are compared.
- [x] Grouped split is the kept result.
- [x] Base rates appear next to metrics.
- [x] Final features contain no future-window, label-derived, or product-flag inputs.
- [x] Deliberate leakage test is shown and excluded.
- [x] Final claim is observed / measured / directional / decision-support.
